# Flight Delay Analysis and Prediction Using Big Data Analytics
### Academic BDA CBP Project using PySpark, MLlib, and Streamlit

This notebook demonstrates:
1. Distributed data loading using **PySpark**
2. Data cleaning, cancellation filtering, and feature engineering
3. Big Data analytics and aggregations (`groupBy`, `agg`, `orderBy`)
4. Machine Learning classification using **PySpark MLlib RandomForestClassifier**
5. Real-time inference using the persisted MLlib PipelineModel

In [ ]:
import sys, os
from pathlib import Path
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root))

from src.spark_utils import get_spark_session
from src.data_loader import load_raw_data
from src.data_preprocessing import preprocess_flight_data
from src.analysis import run_flight_analytics
from src.train_model import train_and_evaluate_model
from src.prediction import predict_flight_delay

spark = get_spark_session('FlightDelayNotebook')
print('PySpark Initialized. Spark Version:', spark.version)

## 1. Load Raw Flight Data with PySpark

In [ ]:
df_raw, _ = load_raw_data(spark)
print('Total Raw Records Count:', df_raw.count())
df_raw.printSchema()
df_raw.show(5)

## 2. Data Cleaning & Feature Engineering

In [ ]:
df_clean = preprocess_flight_data(df_raw, save_processed=False)
df_clean.select('Carrier', 'AirlineName', 'OriginAirportName', 'DestAirportName', 'DepartureHour', 'DepDelay', 'IsDelayed').show(5)

## 3. PySpark Big Data Analytics

In [ ]:
analytics_res = run_flight_analytics(df_clean, charts_dir='../outputs/charts')
print('Overall Flight KPIs:', analytics_res['kpis'])

## 4. PySpark MLlib Random Forest Model Training

In [ ]:
metrics = train_and_evaluate_model(df_clean, model_save_dir='../models/flight_delay_model', reports_dir='../outputs/reports')
print('Test Set Accuracy:', metrics['accuracy'])
print('Test Set Weighted Precision:', metrics['precision'])
print('Test Set Weighted Recall:', metrics['recall'])
print('Test Set F1-Score:', metrics['f1_score'])
print('Area Under ROC (AUC):', metrics['roc_auc'])

## 5. Live Interactive Prediction

In [ ]:
sample_pred = predict_flight_delay(
    carrier='DL',
    origin_airport='Hartsfield-Jackson Atlanta International',
    dest_airport='Los Angeles International',
    departure_hour=18,
    day_of_week=5,
    month=7,
    day_of_month=15,
    model_path='../models/flight_delay_model'
)
for k, v in sample_pred.items():
    print(f'{k:20}: {v}')